# Playground to store a documents coming in as texts in a Pinecone index with namespaces

In [19]:
import logging
import os
import openai
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Pinecone
from pinecone import Pinecone as Pinecone_Client
from dotenv import load_dotenv, find_dotenv
from typing import List, Dict
import global_config

logger = logging.getLogger(global_config.LOG_FILE_NAME)

num_splits = 0


def split_texts(texts, chunk_size=10, chunk_overlap=10):
    logger.info("split_docs with " + str(chunk_size) + " and " + str(chunk_overlap))
    # text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    text_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_text(texts)
    global num_splits
    num_splits = len(docs)
    return docs


def find_long_strings(documents, max_length=40000):
    """Find strings in the array longer than a specified length."""
    return [s for s in documents if len(s) > max_length]


def split_string(input_string, chunk_size=20000):
    """Split a string into chunks of a specified size."""
    return [input_string[i:i + chunk_size] for i in range(0, len(input_string), chunk_size)]


def filter_strings(documents, max_length=40000):
    """Filter strings from the array based on length."""
    return [s for s in documents if len(s) <= max_length]


def do_store_documents(namespace: str, documents: List[Dict[str, str]]):
    embeddings = OpenAIEmbeddings(openai_api_key=os.getenv('OPENAI_API_KEY'))
    all_docs = []
    for document in documents:
        content, filename = document['content'], document['filename']

        docs = split_texts(content)
        long_strings = find_long_strings(docs)
        docs = filter_strings(docs)

        for value in long_strings:
            splitted_strings = split_string(value)
            docs += splitted_strings

        source = f"[[Source: {filename}]]"
        for idx, doc in enumerate(docs):
            docs[idx] = f"{source} {doc}"

        # Collect all docs for batch processing
        all_docs += docs

    # Generate embeddings in a batch for all_docs
    pc = Pinecone_Client(api_key=os.getenv('PINECONE_API_KEY'))
    Pinecone.from_texts(all_docs, embeddings, index_name="testindex", namespace=namespace)

    logger.info(f"Stored {len(all_docs)} documents in index {global_config.PINECONE_INDEX}")
    return len(all_docs)


In [20]:
do_store_documents('case_1', [{'content': 'file1content', 'filename' : 'file1name'}, {'content': 'file2content', 'filename' : 'file2name'}])

2